# Experimentos del proyecto: emergencias por cantón

Registro en MLflow de los modelos de la Semana 10 para predecir `obj_n_emergencias_t7`, las emergencias de un cantón dentro de 7 días. La preparación y la división (por fecha) se importan de `pipeline_proyecto.py` (Semana 9, Sesión 2); este notebook no las copia.

Cada corrida registra los hiperparámetros, cómo se dividieron los datos, la huella del archivo, el MAE en entrenamiento y en prueba, los segundos de entrenamiento y el pipeline entrenado.

In [1]:
import hashlib
import logging
import sys
import time

import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

sys.path.insert(0, "../../semana-09-sesion-02/plantilla-pipeline")
from pipeline_proyecto import CORTE, OBJETIVO, RUTA_DATOS, cargar_y_dividir, construir_preparacion
from linea_base import IgualQueHoy

logging.getLogger("mlflow").setLevel(logging.ERROR)

# n_emergencias solo la usa la linea base; la preparacion la descarta.
X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir(columnas_extra=["n_emergencias"])
print("Entrenamiento:", X_ent.shape, "Prueba:", X_prueba.shape)

Entrenamiento: (321900, 8) Prueba: (81030, 8)


## Paso 1: la huella de los datos y el experimento

In [2]:
def huella(ruta):
    with open(ruta, "rb") as archivo:
        return hashlib.md5(archivo.read()).hexdigest()[:12]


huella_datos = huella(RUTA_DATOS)
print("Huella de obt_emergencias_canton_dia.csv:", huella_datos)

EXPERIMENTO = "emergencias-cantones"
print("Registro de MLflow:", mlflow.get_tracking_uri())
mlflow.set_experiment(EXPERIMENTO)

Huella de obt_emergencias_canton_dia.csv: 7624e6f357a7
Registro de MLflow: http://127.0.0.1:5050


<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1794174827462, experiment_id='3', last_update_time=1794174827462, lifecycle_stage='active', name='emergencias-cantones', tags={}>

## Paso 2: entrenar y registrar los modelos de la Semana 10

In [3]:
def evaluar(nombre, modelo):
    # Los modelos se entrenan con el objetivo en log y se miden en la escala
    # original. La linea base no aprende nada: es la regla, sin preparacion.
    if isinstance(modelo, IgualQueHoy):
        pipeline = modelo
        en_log = False
    else:
        pipeline = Pipeline([
            ("preparar", construir_preparacion()),
            ("modelo", TransformedTargetRegressor(regressor=modelo, func=np.log1p, inverse_func=np.expm1)),
        ])
        en_log = True

    with mlflow.start_run(run_name=nombre, tags={"mlflow.source.name": "experimentos_proyecto.ipynb"}):
        mlflow.log_params({
            **modelo.get_params(),
            "corte_division": str(CORTE.date()),
            "separacion_dias": 7,
            "huella_datos": huella_datos,
            "filas_entrenamiento": len(X_ent),
            "objetivo_en_log": en_log,
        })
        inicio = time.perf_counter()
        pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio
        mae_ent = mean_absolute_error(y_ent, pipeline.predict(X_ent))
        mae_prueba = mean_absolute_error(y_prueba, pipeline.predict(X_prueba))
        mlflow.log_metrics({"mae_entrenamiento": mae_ent, "mae_prueba": mae_prueba,
                            "brecha": mae_prueba - mae_ent, "segundos": segundos})
        mlflow.sklearn.log_model(pipeline, "modelo")
    print(f"{nombre:32s} entrenamiento {mae_ent:8.4f}   prueba {mae_prueba:8.4f}   {segundos:6.1f} s")

In [4]:
evaluar("linea base (igual que hoy)", IgualQueHoy())
evaluar("ridge alpha=100", Ridge(alpha=100.0))
evaluar("gradient boosting max_depth=5", GradientBoostingRegressor(max_depth=5, random_state=42))
evaluar("red neuronal (64)", MLPRegressor(hidden_layer_sizes=(64,), max_iter=200, early_stopping=True, random_state=42))

linea base (igual que hoy)       entrenamiento   6.9318   prueba   6.7778      0.0 s


ridge alpha=100                  entrenamiento   6.2442   prueba   6.0395      0.3 s


gradient boosting max_depth=5    entrenamiento   5.5520   prueba   5.5490     23.5 s


red neuronal (64)                entrenamiento   5.6595   prueba   5.5009     10.1 s


## Paso 3: el registro, la elección y la reproducción

In [5]:
experimento = mlflow.get_experiment_by_name(EXPERIMENTO)
corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id], order_by=["metrics.mae_prueba ASC"])
columnas = ["run_id", "tags.mlflow.runName", "metrics.mae_entrenamiento", "metrics.mae_prueba",
            "metrics.brecha", "metrics.segundos", "params.corte_division", "params.huella_datos"]
print(corridas[columnas].to_string(index=False))

                          run_id           tags.mlflow.runName  metrics.mae_entrenamiento  metrics.mae_prueba  metrics.brecha  metrics.segundos params.corte_division params.huella_datos
79a9fa5ad0704df9a55e85776fca0252             red neuronal (64)                   5.659503            5.500892       -0.158611         10.138748            2025-08-25        7624e6f357a7
05dac271ba674e7c967cbc31c5da8989 gradient boosting max_depth=5                   5.551985            5.548994       -0.002990         23.508555            2025-08-25        7624e6f357a7
f49451a07c8544ca9ab00e52ab0bf0a3               ridge alpha=100                   6.244183            6.039505       -0.204678          0.298673            2025-08-25        7624e6f357a7
9d4ac24fc7094926b67e246bb82e6447    linea base (igual que hoy)                   6.931783            6.777761       -0.154022          0.000003            2025-08-25        7624e6f357a7


In [6]:
# La red y el gradient boosting quedan casi empatados en prueba; elijo el
# gradient boosting porque se puede explicar (importancias) y es determinista.
FINAL = "gradient boosting max_depth=5"
final = corridas[corridas["tags.mlflow.runName"] == FINAL].iloc[0]
final_run_id = final["run_id"]
mlflow.MlflowClient().set_tag(final_run_id, "candidato", "final")

modelo_final = mlflow.sklearn.load_model("runs:/" + final_run_id + "/modelo")
mae_reproducido = mean_absolute_error(y_prueba, modelo_final.predict(X_prueba))
print("Corrida final:", FINAL, final_run_id)
print("MAE registrado:", final["metrics.mae_prueba"], "| MAE reproducido:", mae_reproducido)
print("Reproduce el registro:", abs(mae_reproducido - final["metrics.mae_prueba"]) < 1e-9)

Corrida final: gradient boosting max_depth=5 05dac271ba674e7c967cbc31c5da8989
MAE registrado: 5.548994361235862 | MAE reproducido: 5.548994361235862
Reproduce el registro: True


In [7]:
# La tabla de corridas, ya con la etiqueta candidato = final.
corridas = mlflow.search_runs(experiment_ids=[experimento.experiment_id], order_by=["metrics.mae_prueba ASC"])
corridas.to_csv("corridas_proyecto.csv", index=False)
print("corridas_proyecto.csv:", corridas.shape)
print(corridas[["tags.mlflow.runName", "metrics.mae_prueba", "tags.candidato"]].to_string(index=False))

corridas_proyecto.csv: (4, 61)
          tags.mlflow.runName  metrics.mae_prueba tags.candidato
            red neuronal (64)            5.500892           None
gradient boosting max_depth=5            5.548994          final
              ridge alpha=100            6.039505           None
   linea base (igual que hoy)            6.777761           None
